# Step 1.4 Create a persistent variable review table
* "hrs variable review"

Let's now turn the successful join into a reusable review workflow. We'll create "hrs_variable_review" table.

The table will have the full page_text.

The table will retain the full source page, track review decisions, and let you rerun discovery without resetting previously approved or rejected records.

### How the review process works

|Field	|Purpose|
| --- | --- |
|candidate_token	|Potential HRS variable discovered in the Codebook|
|source_page	|PDF page containing the evidence|
|page_text	|Complete extracted page text|
|review_status	|Pending, Approved, or Rejected|
|review_notes	|Your comments or reasons for the decision|
|reviewed_by, reviewed_at	|Review audit trail|

The merge uses the combination of document, Codebook version, page, and token as its logical key. Delta tables do not automatically enforce uniqueness for that combination, so we deduplicate the incoming data before merging.

### Important: 
This first version stores full page text with each candidate. That is convenient for review, but it can increase storage because multiple candidates may refer to the same page. We can optimize that later if needed.

The candidate finder is identifying words in the Codebook that are not HRS variables.

For example, BIRNBAUM and BROOKE are author names on page 1. They should not become approved variable metadata.

This is expected from a broad token-discovery process. We'll improve the workflow by grouping candidates before asking you to review them, while preserving the source evidence.

### Create the Delta Table

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
catalog_schema = "dev_catalog.hrs_agent_metadata"

candidates_table = f"{catalog_schema}.hrs_variable_candidates"
pages_table = f"{catalog_schema}.hrs_document_pages"
review_table = f"{catalog_schema}.hrs_variable_review"

# ---------------------------------------------------------
# 1. Create the persistent review table if necessary
# ---------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {review_table} (
    -- Candidate Token Details
    document_id       STRING            COMMENT 'Source document identifier',
    codebook_version  STRING            COMMENT 'Source document version',
    source_page       INT               COMMENT 'Source page',
    candidate_token   STRING            COMMENT 'Candidate being reviewed',
    context_excerpt   STRING            COMMENT 'Supporting Codebook text',
    document_title    STRING            COMMENT 'Document title',
    page_text         STRING            COMMENT 'Full text of the source page, if retained here',
    review_status     STRING            COMMENT 'E.g. PENDING, APPROVED, REJECTED',
    review_notes      STRING            COMMENT 'Review comments or explanation',
    -- Candidate classification
    bronze_column_match BOOLEAN         COMMENT 'Whether the candidate matches a Bronze column',
    candidate_class      STRING         COMMENT 'LIKELY_VARIABLE, LIKELY_FALSE_POSITIVE, or NEEDS_REVIEW',
    classification_reason STRING        COMMENT 'Reason for classification',
    classified_at        TIMESTAMP      COMMENT 'Classification timestamp',
    -- Audit Columns
    reviewed_by       STRING            COMMENT 'Reviewer identifier',
    reviewed_at       TIMESTAMP         COMMENT 'Review timestamp',
    created_at        TIMESTAMP         COMMENT 'Creation timestamp',
    updated_at        TIMESTAMP         COMMENT 'Last update timestamp'
)
USING DELTA
""")



### Get the latest pages from HRS Document Pages

In [0]:
# ---------------------------------------------------------
# 2. Read source tables
# ---------------------------------------------------------
candidates = spark.table(candidates_table)
pages = spark.table(pages_table)

# ---------------------------------------------------------
# Keep the most recently extracted copy of each Codebook page.
# ---------------------------------------------------------

# 2.1 Create a data frame "page_window"
#     Partition (segment/group) the data by "document_id", "codebook_version", "page_number".
#     Filters out older duplicate records and cleans up the final output.
page_window = Window.partitionBy(
    "document_id", "codebook_version", "page_number"
).orderBy(F.col("extracted_at").desc_nulls_last())   # "extracted_at" is the timestamp column in hrs_document_pages.

# 2.2 Create a data frame to keep the most recent copy of each page
#     This code deduplicates your dataset to keep only the single most recently extracted version of each page. 
#     It uses the window logic you set up earlier to filter out older duplicate records and cleans up the final output.
latest_pages = (
    pages
    .withColumn("_row_num", F.row_number().over(page_window))  # _row_num is a temporary column used for deduplication
    .filter(F.col("_row_num") == 1)                            # Keep only the rows where the row number is 1
    .drop("_row_num")                                          # Delete the temporary ranking column
    .select(                                                   # Select the data that has been partitioned and deduplicated
        "document_id",
        "codebook_version",
        "page_number",
        "document_title",
        "page_text"
    )
)

# Create a DataFrame to partition/group the candidate data.
candidate_window = Window.partitionBy(
    "document_id",
    "codebook_version",
    "source_page",
    "candidate_token"
).orderBy(F.col("context_excerpt").desc_nulls_last())

# Remove duplicate candidate keys before joining and store in the unique_candidates data frame..
unique_candidates = (
    candidates
    .withColumn("_row_num", F.row_number().over(candidate_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)


### JOIN "candidates" with "pages" to get a full copy of the source page

In [0]:
# ---------------------------------------------------------
# 3. Join each candidate to its full source page
#    JOIN "candidates" with "pages" to get a full copy of the source page
#    Store result in the "source" data frame
# ---------------------------------------------------------
source = (
    unique_candidates.alias("c")                                # Candidates
    .join(                                                      # LEFT OUTER JOIN, preserves all rows from the left table (candidates)
        latest_pages.alias("p"),                                # Pages on the following conditions
        (F.col("c.document_id") == F.col("p.document_id"))      #     ON c.document_id = p.document_id
        & (                                                     #     AND c.codebook_version = p.codebook_version
            F.col("c.codebook_version")                         #     AND c.source_page = p.page_number                    
            == F.col("p.codebook_version")
        )
        & (F.col("c.source_page") == F.col("p.page_number")),
        "left"
    )
    .select(
        F.col("c.document_id").alias("document_id"),
        F.col("c.codebook_version").alias("codebook_version"),
        F.col("c.source_page").alias("source_page"),
        F.col("c.candidate_token").alias("candidate_token"),
        F.col("c.context_excerpt").alias("context_excerpt"),
        F.col("p.document_title").alias("document_title"),
        F.col("p.page_text").alias("page_text")
    )
    .withColumn("source_available", F.col("page_text").isNotNull()) #boolean (True/False) column. If a matching page was successfully
                                                                    # found and merged, it evaluates to True, else False.
)

# Keep only records with an available source page for review.
# cleans your dataset by discarding any records that are missing their underlying text and then drops the validation column
source = source.filter(F.col("source_available")).drop("source_available")

# Register a temporary view for the Delta MERGE.
source.createOrReplaceTempView("hrs_variable_review_source")

### Merge Candidate and Pages data into the Review Table

In [0]:
# ---------------------------------------------------------
# 4. Insert new candidates; refresh evidence on existing rows.
#    Never overwrite review decisions on matched records.
# 
#    review_table = f"{catalog_schema}.hrs_variable_review"
# ---------------------------------------------------------
spark.sql(f"""
MERGE INTO {review_table} AS target                         
USING hrs_variable_review_source AS source
ON  target.document_id = source.document_id
AND target.codebook_version = source.codebook_version
AND target.source_page = source.source_page
AND target.candidate_token = source.candidate_token

WHEN MATCHED THEN UPDATE SET
    target.context_excerpt = source.context_excerpt,
    target.document_title = source.document_title,
    target.page_text = source.page_text,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    document_id,
    codebook_version,
    source_page,
    candidate_token,
    context_excerpt,
    document_title,
    page_text,
    review_status,
    review_notes,
    reviewed_by,
    reviewed_at,
    created_at,
    updated_at
)
VALUES (
    source.document_id,
    source.codebook_version,
    source.source_page,
    source.candidate_token,
    source.context_excerpt,
    source.document_title,
    source.page_text,
    'Pending',
    NULL,
    NULL,
    NULL,
    current_timestamp(),
    current_timestamp()
)
""")

print("Review table:", review_table)
print("Source candidates:", source.count())
print(
    "Candidates with missing source pages:",
    unique_candidates.count() - source.count()
)

display(
    spark.table(review_table)
    .filter(F.col("codebook_version") == "1992-2022v1")
    .orderBy("source_page", "candidate_token")
    .select(
        "candidate_token",
        "source_page",
        "review_status",
        "context_excerpt"
    )
    .limit(25)
)
